# Ingestion — raw API response shapes

One cell per source, calling the **real upstream provider directly**
(`httpx`/the `openelectricity` SDK) — no ported ingestion code involved,
since Phase 1 (`services/ingestion/TODO.md`) hasn't ported the 5
`ingest_*.py` tasks into this service yet. The point here is narrower:
see the *actual raw response shape* each provider returns before writing
the parsing code, same "verified against a real downloaded sample, not
guessed from the spec docs" discipline `data-pipeline`'s own ingest
tasks already used.

Each cell is wrapped in its own `try/except` — one source's endpoint
being unreachable from wherever this happens to run shouldn't stop the
others from showing real output.

In [1]:
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

# This notebook lives at services/ingestion/notebooks/ -- walk up to find
# services/ingestion/.env regardless of the kernel's actual cwd, same
# "don't assume a fixed relative offset" reasoning as data-pipeline's
# notebooks' own setup cells.
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
INGESTION_DIR = next((c for c in _candidates if (c / ".env").exists()), None)
assert INGESTION_DIR is not None, (
    f"couldn't find services/ingestion/.env starting from cwd={Path.cwd()}"
)
load_dotenv(INGESTION_DIR / ".env")

print("loaded .env from:", INGESTION_DIR / ".env")
print("OE_API_KEY set:", bool(os.environ.get("OE_API_KEY")))

loaded .env from: /Users/macbook/Project/research/EcoLens/services/ingestion/.env
OE_API_KEY set: True


In [2]:
# fetch aemo-nem — real public Archive (no auth), nemweb.com.au
# `PUBLIC_DISPATCHIS_{YYYYMMDD}.zip` -- a zip-of-288-nested-zips, one per
# real 5-min dispatch interval. Same URL data-pipeline's
# `ingest_aemo_nem.py._fetch_archive_day` uses for real historical/
# backfill fetches (its near-real-time "live" tier is a placeholder that
# never actually parses a response -- see docs/data/ingestion.md).
!uv add httpx
import io
import zipfile
from datetime import date, timedelta

day = date.today() - timedelta(days=2)  # give the Archive time to publish
url = (
    "https://www.nemweb.com.au/Reports/ARCHIVE/DispatchIS_Reports/"
    f"PUBLIC_DISPATCHIS_{day.strftime('%Y%m%d')}.zip"
)

try:
    async with httpx.AsyncClient(
        timeout=60, headers={"User-Agent": "Mozilla/5.0"}
    ) as client:
        resp = await client.get(url)
    resp.raise_for_status()
    print("url:", url)
    print("status:", resp.status_code, "| bytes:", len(resp.content))

    outer = zipfile.ZipFile(io.BytesIO(resp.content))
    names = outer.namelist()
    print(f"outer zip: {len(names)} nested per-interval zips, e.g. {names[:3]}")

    inner_bytes = outer.read(names[0])
    inner = zipfile.ZipFile(io.BytesIO(inner_bytes))
    csv_name = next(n for n in inner.namelist() if n.upper().endswith(".CSV"))
    csv_text = inner.read(csv_name).decode("utf-8", errors="replace")
    print(f"\none nested zip -> {csv_name!r}, first 6 lines of the real MMS CSV:")
    for line in csv_text.splitlines()[:6]:
        print(" ", line)
except Exception as exc:
    print(f"aemo-nem fetch failed ({type(exc).__name__}): {exc}")

Resolved 183 packages in 6ms


Audited 75 packages in 3ms


url: https://www.nemweb.com.au/Reports/ARCHIVE/DispatchIS_Reports/PUBLIC_DISPATCHIS_20260803.zip
status: 200 | bytes: 5686207
outer zip: 288 nested per-interval zips, e.g. ['PUBLIC_DISPATCHIS_202608030005_0000000530591757.zip', 'PUBLIC_DISPATCHIS_202608030010_0000000530592271.zip', 'PUBLIC_DISPATCHIS_202608030015_0000000530592833.zip']

one nested zip -> 'PUBLIC_DISPATCHIS_202608030005_0000000530591757.CSV', first 6 lines of the real MMS CSV:
  C,NEMP.WORLD,DISPATCHIS,AEMO,PUBLIC,2026/08/03,00:00:08,0000000530591757,DISPATCHIS,0000000530591756
  I,DISPATCH,CASE_SOLUTION,2,SETTLEMENTDATE,RUNNO,INTERVENTION,CASESUBTYPE,SOLUTIONSTATUS,SPDVERSION,NONPHYSICALLOSSES,TOTALOBJECTIVE,TOTALAREAGENVIOLATION,TOTALINTERCONNECTORVIOLATION,TOTALGENERICVIOLATION,TOTALRAMPRATEVIOLATION,TOTALUNITMWCAPACITYVIOLATION,TOTAL5MINVIOLATION,TOTALREGVIOLATION,TOTAL6SECVIOLATION,TOTAL60SECVIOLATION,TOTALASPROFILEVIOLATION,TOTALFASTSTARTVIOLATION,TOTALENERGYOFFERVIOLATION,LASTCHANGED,SWITCHRUNINITIALSTATUS,SWITCH

In [3]:
# fetch aemo-wem — real public data portal (no auth), data.wa.aemo.com.au
# Two separate real endpoints for one day (data-pipeline's
# `ingest_aemo_wem.py._fetch_wem_day`): demand (JSON, every 5-min
# interval) and price (a zip containing JSON, 30-min only). Just the
# demand endpoint here -- enough to see the real response shape.
from datetime import date, timedelta

day = date.today() - timedelta(days=2)
url = (
    "https://data.wa.aemo.com.au/public/market-data/wemde/"
    f"operationalDemandWithdrawal/dailyFiles/OperationalDemandAndWithdrawal_{day.isoformat()}.json"
)

try:
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url)
    resp.raise_for_status()
    payload = resp.json()
    print("url:", url)
    print("status:", resp.status_code)
    print("top-level keys:", list(payload.keys()))

    records = payload.get("data", {}).get("data", [])
    print(f"\n{len(records)} demand records; first one:")
    print(records[0] if records else None)
except Exception as exc:
    print(f"aemo-wem fetch failed ({type(exc).__name__}): {exc}")

url: https://data.wa.aemo.com.au/public/market-data/wemde/operationalDemandWithdrawal/dailyFiles/OperationalDemandAndWithdrawal_2026-08-03.json
status: 200
top-level keys: ['data', 'errors', 'warnings', 'infos', 'transactionId']

288 demand records; first one:
{'dispatchInterval': '2026-08-03T08:00:00+08:00', 'asAtTimeStamp': '2026-08-03T08:05:00+08:00', 'operationalDemand': 2811.36865, 'operationalWithdrawal': -7.22192}


In [4]:
# fetch oe — real OpenElectricity SDK call (needs OE_API_KEY)
# The one source with no fallback tiers at all in data-pipeline's own
# ingest task -- every real fetch goes through this SDK method
# (`app/service/emissions.py._fetch_metric`). `date_start`/`date_end`
# must be naive, in the network's own local time (OE rejects tz-aware
# values) -- NEM is fixed UTC+10, no DST.
from datetime import datetime, timedelta, timezone

from openelectricity import AsyncOEClient, DataMetric

_NEM_TZ = timezone(timedelta(hours=10))
since_utc = datetime.now(timezone.utc) - timedelta(hours=1)
since_naive_local = since_utc.astimezone(_NEM_TZ).replace(tzinfo=None)

try:
    async with AsyncOEClient(api_key=os.environ["OE_API_KEY"]) as client:
        response = await client.get_network_data(
            network_code="NEM",
            metrics=[DataMetric.POWER],
            date_start=since_naive_local,
            network_region="NSW1",
            secondary_grouping="fueltech",
        )

    records = response.to_records()
    print(f"{len(records)} records for NSW1, last 1h")
    if records:
        print("\nfirst record:", records[0])
        print("\nfields on one record:", list(records[0].keys()))
        print("\ndistinct fuel_type values seen:", {r.get("fueltech") for r in records})
except Exception as exc:
    print(f"oe fetch failed ({type(exc).__name__}): {exc}")

[2026-08-05 17:57:06] DEBUG [openelectricity.client.__init__:150] Initialized client with base URL: https://api.openelectricity.org.au/v4/


[2026-08-05 17:57:06] DEBUG [openelectricity.client.__init__:594] Initialized asynchronous client


[2026-08-05 17:57:06] DEBUG [openelectricity.client._ensure_client:599] Creating new async client session


[2026-08-05 17:57:06] DEBUG [openelectricity.client.get_network_data:672] Getting network data for NEM (metrics: [<DataMetric.POWER: 'power'>], interval: None)


[2026-08-05 17:57:06] DEBUG [openelectricity.client.get_network_data:692] Request parameters: {'metrics': ['power'], 'date_start': '2026-08-05T20:57:06.083973', 'network_region': 'NSW1', 'secondary_grouping': 'fueltech'}


[2026-08-05 17:57:08] DEBUG [openelectricity.client._handle_response:612] Received successful response: 200


[2026-08-05 17:57:08] DEBUG [openelectricity.client.close:787] Closing async client session


120 records for NSW1, last 1h

first record: {'interval': datetime.datetime(2026, 8, 6, 7, 0), 'fueltech': 'battery', 'power': 560.0234}

fields on one record: ['interval', 'fueltech', 'power']

distinct fuel_type values seen: {'hydro', 'battery', 'wind', 'bioenergy_biomass', 'distillate', 'gas_ccgt', 'battery_charging', 'battery_discharging', 'solar_utility', 'pumps', 'coal_black', 'gas_ocgt'}


In [5]:
# fetch holidays — no external API at all
# `ingest_holidays.py.run()` never makes an HTTP call -- it builds a
# small (region, date, holiday_name, is_workday) table in memory from a
# static list in code (`_BASE_HOLIDAYS`). Included here for contrast with
# the 4 real-API sources above, not because there's a response to inspect.
_BASE_HOLIDAYS = (
    ("New Year's Day", "01-01"),
    ("Australia Day", "01-26"),
    ("Good Friday", "varies"),  # would be computed from Easter
    ("Easter Monday", "varies"),
    ("Anzac Day", "04-25"),
    ("Christmas Day", "12-25"),
    ("Boxing Day", "12-26"),
)

print("holidays has no HTTP response to inspect -- it's a static, in-code table:")
for name, month_day in _BASE_HOLIDAYS:
    print(f"  {name}: --{month_day}")

holidays has no HTTP response to inspect -- it's a static, in-code table:
  New Year's Day: --01-01
  Australia Day: --01-26
  Good Friday: --varies
  Easter Monday: --varies
  Anzac Day: --04-25
  Christmas Day: --12-25
  Boxing Day: --12-26


In [6]:
# fetch bom — real live public JSON (no auth), bom.gov.au
# `ingest_bom.py`'s live tier is genuinely real (unlike AEMO's), one call
# per station. Just NSW1's station here (066037, Sydney Airport) --
# `app.core.config.Settings.bom_stations` has all 6.
station_id = "066037"  # NSW1 -- Sydney Airport
url = f"http://www.bom.gov.au/fwo/{station_id}/observations.json"

try:
    async with httpx.AsyncClient(timeout=10) as client:
        resp = await client.get(url)
    resp.raise_for_status()
    payload = resp.json()
    print("url:", url)
    print("status:", resp.status_code)
    print("top-level keys:", list(payload.keys()))

    obs = payload.get("observations", {}).get("data", [])
    print(f"\n{len(obs)} observations; most recent record:")
    print(obs[0] if obs else None)
except Exception as exc:
    print(f"bom fetch failed ({type(exc).__name__}): {exc}")

bom fetch failed (ConnectTimeout): 
